In [3]:
"""
Embedding Entendimiento 
-----------------------
Basicamente tengo en posicion 0 = # Eventos 
+ 10 mas populares categorias (para rolling dias 1-7-30)
+ 10 mas populares skus (para rolling dias 1-7-30)
+ 10 mas populares price bucket (para rolling dias 1-7-30)

Total 1 + (10+10+10) + (10+10+10) + (10+10+10) = 91 dimention
   events   category.     skus.         price
      30    1-7-30        1-7-30        1-7-30    days
      
"""

'\nEmbedding Entendimiento \n-----------------------\nBasicamente tengo en posicion 0 = # Eventos \n+ 10 mas populares categorias (para rolling dias 1-7-30)\n+ 10 mas populares skus (para rolling dias 1-7-30)\n+ 10 mas populares price bucket (para rolling dias 1-7-30)\n\nTotal 1 + (10+10+10) + (10+10+10) + (10+10+10) = 91 dimention\n   events   category.     skus.         price\n      30    1-7-30        1-7-30        1-7-30    days\n      \n'

In [4]:
import argparse
import logging
from typing import List, Tuple
from pathlib import Path
import pandas as pd
import numpy as np

In [5]:
cd ..

/teamspace/studios/this_studio/recsys2025


/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
from baseline.aggregated_features_baseline.constants import (
    EVENT_TYPE_TO_COLUMNS,
)
from data_utils.utils import (
    load_with_properties,
)
from data_utils.data_dir import DataDir
from baseline.aggregated_features_baseline.features_aggregator import (
    FeaturesAggregator,
)

In [7]:
logging.basicConfig()
logger = logging.getLogger(__name__)
logger.setLevel(level=logging.INFO)

In [8]:
def load_relevant_clients_ids(input_dir: Path) -> np.ndarray:
    return np.load(input_dir / "relevant_clients.npy")

In [9]:
def save_embeddings(
    embeddings_dir: Path, embeddings: np.ndarray, client_ids: np.ndarray
):
    """
    Function creates embeddings directory and saves embeddings in competition entry format.

    Args:
    embeddings_dir (Path): The directory where to save embeddings and client_ids.
    embeddings (np.ndarray): 2-d array storing embeddings.
    client_ids (np.ndarray): 1-d array storing client_ids corresponding to vectors from embeddings array.
    """
    logger.info("Saving embeddings")
    embeddings_dir.mkdir(parents=True, exist_ok=True)
    np.save(embeddings_dir / "embeddings.npy", embeddings)
    np.save(embeddings_dir / "client_ids.npy", client_ids)


def create_embeddings(
    data_dir: DataDir,
    num_days: List[int],
    top_n: int,
    relevant_client_ids: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate and merge user representation embeddings for specified event types.

    This function processes event data from CSV files, aggregates user's events based
    on specified columns for each event type, and merges these embeddings into a single
    user representation.

    Args:
        data_dir (DataDir): The DataDir class where Paths to raw event data, input and targte folders are stored.
        num_days (List[int]): A list of time windows (in days) for generating features.
        Each time window will produce different set of features from aggregated events
        from defined period.
        top_n (int): Number of columns' top values to consider for aggregating events.

    Returns:
        Tuple[np.ndarray, np.ndarray] : generated feature matrix and the list of all
        clients in two np.ndarray's.
    """
    aggregator = FeaturesAggregator(
        num_days=num_days,
        top_n=top_n,
        relevant_client_ids=relevant_client_ids,
    )
    for event_type in EVENT_TYPE_TO_COLUMNS.keys():
        logger.info("Generating features for %s event type", event_type.value)
        logger.info("Loading data...")
        event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)
        event_df["timestamp"] = pd.to_datetime(event_df.timestamp)
        logger.info("Generating features...")
        aggregator.generate_features(
            event_type=event_type,
            client_id_column="client_id",
            df=event_df,
            columns=EVENT_TYPE_TO_COLUMNS[event_type],
        )

    logger.info("Merging features into embeddings")
    client_ids, embeddings = aggregator.merge_features()
    return client_ids, embeddings


In [10]:
def get_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data-dir",
        type=str,
        required=True,
        help="Directory with input and target data – produced by data_utils.split_data",
    )
    parser.add_argument(
        "--embeddings-dir",
        type=str,
        required=True,
        help="Directory where to store generated embeddings",
    )
    parser.add_argument(
        "--num-days",
        nargs="*",
        type=int,
        default=[1, 7, 30],
        help="Numer of days to compute features",
    )
    parser.add_argument(
        "--top-n",
        type=int,
        default=10,
        help="Number of top column values to consider in feature generation",
    )
    return parser

In [11]:
data_dir = "/teamspace/studios/this_studio/ubc_data"
data_dir = DataDir(Path(data_dir))

In [12]:
embeddings_dir = "/teamspace/studios/this_studio/ubc_data/embeddings"
embeddings_dir = Path(embeddings_dir)

In [13]:
relevant_client_ids = load_relevant_clients_ids(input_dir=data_dir.input_dir)

In [14]:
relevant_client_ids

array([ 5963217, 17797869, 18408314, ..., 22423862, 16846917, 14884775])

In [15]:
############ DRAFT DEBUG ########################################

In [334]:
######### Debug create embeddings 

num_days=[1, 7, 30]
top_n=10

aggregator = FeaturesAggregator(
    num_days=num_days,
    top_n=top_n,
    relevant_client_ids=relevant_client_ids,
)

In [335]:
for event_type in EVENT_TYPE_TO_COLUMNS.keys():
    break

In [336]:
event_type

<EventTypes.PRODUCT_BUY: 'product_buy'>

In [337]:
event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)

In [2]:
data_dir

NameError: name 'data_dir' is not defined

In [338]:
event_df

,client_id,timestamp,sku,category,price,name
0,17649961,2022-07-23 20:15:25,18485,5492,72,[187 47 120 237 234 172 91 172 67 153 32 ...
1,16696114,2022-07-11 16:31:30,81192,6519,99,[241 241 241 241 241 241 241 241 112 241 241 2...
2,10238779,2022-05-29 19:35:40,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
3,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
4,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
...,...,...,...,...,...,...
1315056,7535041,2022-08-19 04:58:40,269962,2895,14,[128 159 53 89 131 143 53 171 53 214 141 1...
1315057,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...
1315058,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...
1315059,20304565,2022-09-02 12:38:50,79597,5349,0,[196 97 71 127 98 174 234 70 25 2 202 2...


In [339]:
event_df["timestamp"] = pd.to_datetime(event_df.timestamp)

In [19]:
# Debg aggregator.generate_features

In [340]:
self = aggregator

In [341]:
EVENT_TYPE_TO_COLUMNS[event_type]

['sku', 'category', 'price']

In [342]:
# Parameters 
#event_type=event_type
client_id_column="client_id"
df=event_df
columns=EVENT_TYPE_TO_COLUMNS[event_type]

In [343]:
columns

['sku', 'category', 'price']

In [344]:
df = self._filter_events_to_relevant_clients(df)

In [345]:
calculator = self.get_calculator(
    event_type=event_type,
    df=df,
    columns=columns,
)

In [346]:
calculator._max_date

Timestamp('2022-09-12 23:58:35')

In [347]:
self._update_features_sizes(
            event_type=event_type, features_size=calculator.features_size
        )

In [348]:
df

,client_id,timestamp,sku,category,price,name
0,17649961,2022-07-23 20:15:25,18485,5492,72,[187 47 120 237 234 172 91 172 67 153 32 ...
1,16696114,2022-07-11 16:31:30,81192,6519,99,[241 241 241 241 241 241 241 241 112 241 241 2...
2,10238779,2022-05-29 19:35:40,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
3,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
4,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
...,...,...,...,...,...,...
1315052,22420656,2022-06-04 14:28:05,82693,1051,78,[237 247 201 218 198 218 14 2 2 237 218 ...
1315053,9661121,2022-06-10 08:30:25,149624,4981,76,[219 61 242 13 243 165 13 97 13 90 145 ...
1315054,23073613,2022-06-14 14:07:50,659157,2864,13,[180 42 251 192 218 31 54 170 199 40 16 ...
1315057,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...


In [349]:
len(df.category.unique())

5706

In [350]:
len(df.sku.unique())

331795

In [351]:
ans = df.groupby(client_id_column)

In [352]:
from tqdm import tqdm


In [353]:
 for client_id, events in tqdm(df.groupby(client_id_column)):
    if client_id == 13684077:
      break

 57%|█████▋    | 238744/416269 [00:04<00:03, 55763.27it/s]


In [354]:
events

,client_id,timestamp,sku,category,price,name
616262,13684077,2022-09-08 21:42:55,436878,1517,36,[100 109 187 47 26 47 102 110 238 216 13 1...
616263,13684077,2022-09-08 21:42:55,39041,1374,86,[121 65 41 254 55 47 127 241 193 244 6 2...
616264,13684077,2022-09-08 21:42:55,814355,1374,88,[209 26 93 178 233 47 127 110 41 244 202 2...
616265,13684077,2022-09-08 21:42:55,774895,139,36,[ 15 6 147 9 147 47 138 212 212 54 167 ...
616266,13684077,2022-09-08 21:42:55,1478935,831,80,[152 21 110 41 127 47 138 110 109 110 142 2...
...,...,...,...,...,...,...
616922,13684077,2022-09-12 01:38:45,597704,2624,92,[ 94 23 140 249 78 190 140 140 161 84 198 1...
616923,13684077,2022-09-12 01:38:45,371140,2624,86,[ 94 185 140 249 78 190 140 140 155 84 198 1...
616924,13684077,2022-09-12 01:38:45,608165,831,77,[152 109 132 224 194 47 140 158 223 54 13 1...
616925,13684077,2022-09-12 01:38:45,533529,831,86,[ 15 109 110 47 110 47 138 38 223 54 142 1...


In [355]:
counts = df.groupby(client_id_column).size()
top_user = counts.idxmax()
top_user_count = counts.max()

print(f"User with most rows: {top_user} (Rows: {top_user_count})")

User with most rows: 13684077 (Rows: 665)


In [356]:
client_id

13684077

In [357]:
events

,client_id,timestamp,sku,category,price,name
616262,13684077,2022-09-08 21:42:55,436878,1517,36,[100 109 187 47 26 47 102 110 238 216 13 1...
616263,13684077,2022-09-08 21:42:55,39041,1374,86,[121 65 41 254 55 47 127 241 193 244 6 2...
616264,13684077,2022-09-08 21:42:55,814355,1374,88,[209 26 93 178 233 47 127 110 41 244 202 2...
616265,13684077,2022-09-08 21:42:55,774895,139,36,[ 15 6 147 9 147 47 138 212 212 54 167 ...
616266,13684077,2022-09-08 21:42:55,1478935,831,80,[152 21 110 41 127 47 138 110 109 110 142 2...
...,...,...,...,...,...,...
616922,13684077,2022-09-12 01:38:45,597704,2624,92,[ 94 23 140 249 78 190 140 140 161 84 198 1...
616923,13684077,2022-09-12 01:38:45,371140,2624,86,[ 94 185 140 249 78 190 140 140 155 84 198 1...
616924,13684077,2022-09-12 01:38:45,608165,831,77,[152 109 132 224 194 47 140 158 223 54 13 1...
616925,13684077,2022-09-12 01:38:45,533529,831,86,[ 15 109 110 47 110 47 138 38 223 54 142 1...


In [358]:
features = calculator.compute_features(events=events)

In [360]:
features

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,  21.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,  83.,   0.,   0.,   0.,
         0.,   0.,  21.,  58.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,  83.,   0.,   0.,   0.,   0.,   0.,  21.,
        58.,   0.,   0.], dtype=float16)

In [74]:
np.sum(features)

np.float16(1010.0)

In [75]:
len(features)

91

In [76]:
calculator._max_date

Timestamp('2022-09-12 23:58:35')

In [77]:
type(calculator)

baseline.aggregated_features_baseline.calculators.StatsFeaturesCalculator

In [57]:
### Debuug calculator StatsFeaturesCalculator 

In [361]:
self = calculator

In [362]:
self.features_size

91

In [363]:
from baseline.aggregated_features_baseline.constants import (
    EMBEDDINGS_DTYPE,
)
from datetime import timedelta

In [364]:
features = np.zeros(self.features_size, dtype=EMBEDDINGS_DTYPE)

In [365]:
features

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0.], dtype=float16)

In [366]:
features[0] = events.shape[0]

In [367]:
pointer = 1
timestamps = events["timestamp"].sort_values()

In [368]:
timestamps

616262   2022-09-08 21:42:55
616263   2022-09-08 21:42:55
616264   2022-09-08 21:42:55
616265   2022-09-08 21:42:55
616266   2022-09-08 21:42:55
                 ...        
616924   2022-09-12 01:38:45
616925   2022-09-12 01:38:45
616922   2022-09-12 01:38:45
616923   2022-09-12 01:38:45
616926   2022-09-12 01:38:45
Name: timestamp, Length: 665, dtype: datetime64[ns]

In [304]:
self._num_days

[1, 7, 30]

In [305]:
self.features_size

91

In [415]:
features = np.zeros(self.features_size, dtype=EMBEDDINGS_DTYPE)
features[0] = events.shape[0]
pointer = 1
timestamps = events["timestamp"].sort_values()

In [389]:
events

,client_id,timestamp,sku,category,price,name
616262,13684077,2022-09-08 21:42:55,436878,1517,36,[100 109 187 47 26 47 102 110 238 216 13 1...
616263,13684077,2022-09-08 21:42:55,39041,1374,86,[121 65 41 254 55 47 127 241 193 244 6 2...
616264,13684077,2022-09-08 21:42:55,814355,1374,88,[209 26 93 178 233 47 127 110 41 244 202 2...
616265,13684077,2022-09-08 21:42:55,774895,139,36,[ 15 6 147 9 147 47 138 212 212 54 167 ...
616266,13684077,2022-09-08 21:42:55,1478935,831,80,[152 21 110 41 127 47 138 110 109 110 142 2...
...,...,...,...,...,...,...
616922,13684077,2022-09-12 01:38:45,597704,2624,92,[ 94 23 140 249 78 190 140 140 161 84 198 1...
616923,13684077,2022-09-12 01:38:45,371140,2624,86,[ 94 185 140 249 78 190 140 140 155 84 198 1...
616924,13684077,2022-09-12 01:38:45,608165,831,77,[152 109 132 224 194 47 140 158 223 54 13 1...
616925,13684077,2022-09-12 01:38:45,533529,831,86,[ 15 109 110 47 110 47 138 38 223 54 142 1...


In [390]:
events.shape[0]

665

In [391]:
features

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.], dtype=float16)

In [392]:
self._columns

['sku', 'category', 'price']

In [393]:
################# DEBUG ###############
features

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.], dtype=float16)

In [394]:
days = 30
start_date = self._max_date - timedelta(days=days)
idx = timestamps.searchsorted(start_date)

In [395]:
column = 'category'

In [396]:
pointer

1

In [397]:
features_to_write = features[
        pointer : pointer + len(self._unique_values[column])
    ]

In [398]:
pointer + len(self._unique_values[column])

11

In [448]:
len(features_to_write)

10

In [400]:
events[column].to_numpy()[idx:]

array([1517, 1374, 1374,  139,  831, 1374,  680, 2910, 1374,  139,    1,
       1374, 1517, 1374, 2910,  831,  139,  680,  139, 1374, 1374,    1,
       1374, 1374, 2910,  139,    1, 1517,  139, 1374,  831,  680, 1374,
          1,  680, 1374, 1374,  831,  139, 1374, 1374,  139, 1517, 2910,
       1374,    1, 1374,  680,  139, 1374, 1374, 1517,  831,  139, 2910,
       1374,  139, 2910,  680, 1374, 1374, 1374,    1, 1517,  831,  139,
       1374, 1374, 2910, 1517,  139, 1374, 1374,  139,  831,  680,    1,
       1517,    1, 1374, 1374,  680, 1374,  139, 1374,  831, 2910,  139,
        680, 1517, 1374,  139, 1374,  139,  831, 1374, 1374, 2910,    1,
        139, 1374, 1374, 1374,  680, 2910,    1, 1517, 1374,  831,  139,
       1374, 1374,  831, 1517, 1374,  139, 1374,  139,  680, 2910,    1,
       1374,  831, 1374, 1374,  139, 1517,    1, 2910, 1374,  139,  680,
       1374, 2910,    1,  139, 1374, 1517, 1374,  139, 1374,  831,  680,
       1374, 1374, 2910,  680, 1517,    1,  139, 13

In [401]:
[column]

['category']

In [402]:
values = events[column].to_numpy()[idx:]

In [403]:
values

array([1517, 1374, 1374,  139,  831, 1374,  680, 2910, 1374,  139,    1,
       1374, 1517, 1374, 2910,  831,  139,  680,  139, 1374, 1374,    1,
       1374, 1374, 2910,  139,    1, 1517,  139, 1374,  831,  680, 1374,
          1,  680, 1374, 1374,  831,  139, 1374, 1374,  139, 1517, 2910,
       1374,    1, 1374,  680,  139, 1374, 1374, 1517,  831,  139, 2910,
       1374,  139, 2910,  680, 1374, 1374, 1374,    1, 1517,  831,  139,
       1374, 1374, 2910, 1517,  139, 1374, 1374,  139,  831,  680,    1,
       1517,    1, 1374, 1374,  680, 1374,  139, 1374,  831, 2910,  139,
        680, 1517, 1374,  139, 1374,  139,  831, 1374, 1374, 2910,    1,
        139, 1374, 1374, 1374,  680, 2910,    1, 1517, 1374,  831,  139,
       1374, 1374,  831, 1517, 1374,  139, 1374,  139,  680, 2910,    1,
       1374,  831, 1374, 1374,  139, 1517,    1, 2910, 1374,  139,  680,
       1374, 2910,    1,  139, 1374, 1517, 1374,  139, 1374,  831,  680,
       1374, 1374, 2910,  680, 1517,    1,  139, 13

In [404]:
self._unique_values[column]

Index([3068, 5954, 3147, 2739, 5476, 6839, 4142, 4981, 3944, 6440], dtype='int64', name='category')

In [405]:
for val in np.unique(values):
        features_to_write[self._unique_values[column] == val] += np.sum(
            values == val
        )

In [406]:
features_to_write

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float16)

In [407]:
np.sum(
            values == val
        )

np.int64(14)

In [428]:
self._unique_values

{'sku': Index([1061794, 1466277, 1420323,   62095,  237442, 1326077,  190359,  343714,
        1225451,  604561],
       dtype='int64', name='sku'),
 'category': Index([3068, 5954, 3147, 2739, 5476, 6839, 4142, 4981, 3944, 6440], dtype='int64', name='category'),
 'price': Index([58, 69, 60, 73, 66, 41, 77, 36, 26, 63], dtype='int64', name='price')}

In [408]:
self._unique_values[column]

Index([3068, 5954, 3147, 2739, 5476, 6839, 4142, 4981, 3944, 6440], dtype='int64', name='category')

In [409]:
features_to_write

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float16)

In [410]:
pointer += len(self._unique_values[column])

In [411]:
pointer

11

In [412]:
features

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.], dtype=float16)

In [182]:
#######################################

In [ ]:
for days in self._num_days:
    start_date = self._max_date - timedelta(days=days)
    idx = timestamps.searchsorted(start_date)

    for column in self._columns:
        features_to_write = features[
            pointer : pointer + len(self._unique_values[column])
        ] # Extract a slice from feeatures ( a part )

        if features_to_write.size > 0 and np.any(features_to_write != 0):
            break

        values = events[column].to_numpy()[idx:]
        for val in np.unique(values):
            features_to_write[self._unique_values[column] == val] += np.sum(
                values == val
            )
        pointer += len(self._unique_values[column])


In [427]:
features_to_write

array([83.,  0.,  0.,  0.,  0.,  0., 21., 58.,  0.,  0.], dtype=float16)

In [423]:
self._unique_values[column] == val

array([False, False, False, False, False, False, False, False, False,
       False])

In [426]:
np.sum(values == val)

np.int64(20)

In [425]:
values == val

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,

In [418]:
pointer

91

In [420]:
pointer + len(self._unique_values[column])

101

In [421]:
values = events[column].to_numpy()[idx:]
values

array([36, 86, 88, 36, 80, 58,  0, 42, 68, 47, 33, 68, 36, 88, 42, 80, 47,
        0, 36, 58, 86, 33, 68, 86, 42, 36, 33, 36, 47, 58, 80,  0, 88, 33,
        0, 68, 88, 80, 47, 58, 86, 36, 36, 42, 88, 33, 58,  0, 47, 68, 86,
       36, 80, 36, 42, 68, 36, 42,  0, 86, 58, 88, 33, 36, 80, 47, 86, 88,
       42, 36, 36, 58, 68, 47, 80,  0, 33, 36, 33, 68, 88,  0, 58, 47, 86,
       80, 42, 36,  0, 36, 88, 47, 68, 36, 80, 86, 58, 42, 33, 47, 88, 86,
       58,  0, 42, 33, 36, 68, 80, 36, 68, 86, 80, 36, 88, 47, 58, 36,  0,
       42, 33, 86, 80, 88, 58, 36, 36, 33, 42, 68, 47,  0, 86, 42, 33, 47,
       88, 36, 68, 36, 58, 80,  0, 68, 86, 42,  0, 36, 33, 36, 58, 88, 80,
       47, 86, 68, 58,  0,  0, 58, 68, 86, 68,  0, 58, 86, 68,  0, 86, 58,
       58, 86,  0, 68,  0, 68, 58, 86, 86,  0, 68, 58, 68, 58, 86,  0, 68,
       58,  0, 86, 86,  0, 58, 68, 58, 68, 86,  0, 86, 68, 58,  0, 86, 68,
       58,  0, 86, 68,  0, 58,  0, 58, 86, 68,  0, 58, 86, 68, 58, 86,  0,
       68, 97, 87, 87, 97

In [422]:
self._unique_values[column] == val

array([False, False, False, False, False, False, False, False, False,
       False])

In [441]:
len(self._unique_values['sku'])

10

In [443]:
len(self._unique_values['category'])

10

In [445]:
len(self._unique_values['price'])

10

In [385]:
self._num_days

[1, 7, 30]

In [449]:
(10+10+10)*3 # eventos + (1 day + 7 days + 30 days) * 3 (3 Categorias)
"""
Top n = 10 categorias mas comunes 
Top n = 10 skus (item_id) mas comunes 
Top n = 10 price (price) mas comunes
"""

90

In [292]:
column

'price'

In [447]:
features

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,  21.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,  83.,   0.,   0.,   0.,
         0.,   0.,  21.,  58.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,  83.,   0.,   0.,   0.,   0.,   0.,  21.,
        58.,   0.,   0.], dtype=float16)

In [104]:
features_to_write

array([83.,  0.,  0.,  0.,  0.,  0., 21., 58.,  0.,  0.], dtype=float16)

In [127]:
features

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.], dtype=float16)

In [62]:
values

array([], dtype=int64)

In [50]:
pointer

1

In [ ]:
    def compute_features(self, events: pd.DataFrame) -> np.ndarray:
        features = np.zeros(self.features_size, dtype=EMBEDDINGS_DTYPE)
        features[0] = events.shape[0]
        pointer = 1
        timestamps = events["timestamp"].sort_values()
        for days in self._num_days:
            start_date = self._max_date - timedelta(days=days)
            idx = timestamps.searchsorted(start_date)
            for column in self._columns:
                features_to_write = features[
                    pointer : pointer + len(self._unique_values[column])
                ]
                values = events[column].to_numpy()[idx:]
                for val in np.unique(values):
                    features_to_write[self._unique_values[column] == val] += np.sum(
                        values == val
                    )
                pointer += len(self._unique_values[column])

In [ ]:
        


        
        for client_id, events in tqdm(df.groupby(client_id_column)):
            assert isinstance(client_id, int)
            features = calculator.compute_features(events=events)
            self._update_features(
                event_type=event_type,
                client_id=client_id,
                features=features,
            )

In [431]:
logger.info("Generating features for %s event type", event_type.value)
logger.info("Loading data...")
event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)
event_df["timestamp"] = pd.to_datetime(event_df.timestamp)
logger.info("Generating features...")

aggregator.generate_features(
    event_type=event_type,
    client_id_column="client_id",
    df=event_df,
    columns=EVENT_TYPE_TO_COLUMNS[event_type],
)

INFO:__main__:Generating features for product_buy event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
100%|██████████| 416269/416269 [04:13<00:00, 1644.35it/s]


In [432]:
event_type

<EventTypes.PRODUCT_BUY: 'product_buy'>

In [ ]:
"""
Embedding Entendimiento 
-----------------------
Basicamente tengo en posicion 0 = # Eventos 
+ 10 mas populares categorias (para rolling dias 1-7-30)
+ 10 mas populares skus (para rolling dias 1-7-30)
+ 10 mas populares price bucket (para rolling dias 1-7-30)

Total 1 + (10+10+10) + (10+10+10) + (10+10+10) = 91 dimention
   events   category.     skus.         price
      30    1-7-30        1-7-30        1-7-30    days
      
"""

In [1]:
1 + (10+10+10) + (10+10+10) + (10+10+10)

91

In [435]:
aggregator._get_features(client_id=13684077, event_type=event_type)

array([665.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,  21.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,  83.,   0.,   0.,   0.,
         0.,   0.,  21.,  58.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,  83.,   0.,   0.,   0.,   0.,   0.,  21.,
        58.,   0.,   0.], dtype=float16)

In [450]:
len(aggregator._get_features(client_id=13684077, event_type=event_type))

91

In [ ]:
logger.info("Merging features into embeddings")
client_ids, embeddings = aggregator.merge_features()

In [429]:
client_ids, embeddings = create_embeddings(
    data_dir=data_dir,
    num_days=[1, 7, 30],
    top_n=10,
    relevant_client_ids=relevant_client_ids,
)

INFO:__main__:Generating features for product_buy event type
INFO:__main__:Loading data...
INFO:__main__:Generating features...
 60%|██████    | 251190/416269 [02:32<01:40, 1650.62it/s]


KeyboardInterrupt: 

In [ ]:
##################################################################

In [27]:
type(client_ids)

numpy.ndarray

In [28]:
len(client_ids)

858489

In [29]:
type(embeddings)

numpy.ndarray

In [30]:
len(embeddings)

858489

In [33]:
len(embeddings[0])

320

In [34]:
embeddings[0]

array([2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [35]:
# TODO: Debug create_embeddings.py 

# TODO: Debug aggregator.generate_features( 

In [36]:
data_dir

In [37]:
##################### DRAFT  ########################

In [ ]:

def create_embeddings(
    data_dir: DataDir,
    num_days: List[int],
    top_n: int,
    relevant_client_ids: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate and merge user representation embeddings for specified event types.

    This function processes event data from CSV files, aggregates user's events based
    on specified columns for each event type, and merges these embeddings into a single
    user representation.

    Args:
        data_dir (DataDir): The DataDir class where Paths to raw event data, input and targte folders are stored.
        num_days (List[int]): A list of time windows (in days) for generating features.
        Each time window will produce different set of features from aggregated events
        from defined period.
        top_n (int): Number of columns' top values to consider for aggregating events.

    Returns:
        Tuple[np.ndarray, np.ndarray] : generated feature matrix and the list of all
        clients in two np.ndarray's.
    """
    aggregator = FeaturesAggregator(
        num_days=num_days,
        top_n=top_n,
        relevant_client_ids=relevant_client_ids,
    )
    
    for event_type in EVENT_TYPE_TO_COLUMNS.keys():
        logger.info("Generating features for %s event type", event_type.value)
        logger.info("Loading data...")
        event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)
        event_df["timestamp"] = pd.to_datetime(event_df.timestamp)
        logger.info("Generating features...")
        aggregator.generate_features(
            event_type=event_type,
            client_id_column="client_id",
            df=event_df,
            columns=EVENT_TYPE_TO_COLUMNS[event_type],
        )

    logger.info("Merging features into embeddings")
    client_ids, embeddings = aggregator.merge_features()
    return client_ids, embeddings


In [57]:
EVENT_TYPE_TO_COLUMNS.keys()

dict_keys([<EventTypes.PRODUCT_BUY: 'product_buy'>, <EventTypes.ADD_TO_CART: 'add_to_cart'>, <EventTypes.REMOVE_FROM_CART: 'remove_from_cart'>, <EventTypes.PAGE_VISIT: 'page_visit'>, <EventTypes.SEARCH_QUERY: 'search_query'>])

In [72]:
i = 0 

for event_type in EVENT_TYPE_TO_COLUMNS.keys():

    print(event_type)
    if i == 0: 
        break
    i = i + 1

EventTypes.PRODUCT_BUY


In [73]:
event_type

<EventTypes.PRODUCT_BUY: 'product_buy'>

In [74]:
event_type.value

'product_buy'

In [75]:
aggregator = FeaturesAggregator(
        num_days=num_days,
        top_n=top_n,
        relevant_client_ids=relevant_client_ids,
    )

In [76]:
event_df = load_with_properties(data_dir=data_dir, event_type=event_type.value)

In [77]:
event_df

,client_id,timestamp,sku,category,price,name
0,17649961,2022-07-23 20:15:25,18485,5492,72,[187 47 120 237 234 172 91 172 67 153 32 ...
1,16696114,2022-07-11 16:31:30,81192,6519,99,[241 241 241 241 241 241 241 241 112 241 241 2...
2,10238779,2022-05-29 19:35:40,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
3,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
4,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
...,...,...,...,...,...,...
1315056,7535041,2022-08-19 04:58:40,269962,2895,14,[128 159 53 89 131 143 53 171 53 214 141 1...
1315057,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...
1315058,4284680,2022-08-22 20:48:45,368528,3147,94,[161 161 161 161 141 36 161 161 36 161 161 1...
1315059,20304565,2022-09-02 12:38:50,79597,5349,0,[196 97 71 127 98 174 234 70 25 2 202 2...


In [78]:

event_df["timestamp"] = pd.to_datetime(event_df.timestamp)

In [ ]:
# TODO: Debuggear generate_features dentro de FeaturesAggregator 